### Aqui vamos ver o uso de algumas bibliotecas de python para auxiliar na geração de dados 

In [ ]:
from faker import Faker
import pandas as pd
import os
from datetime import date, timedelta
import numpy as np
from num2words import num2words as nw


ModuleNotFoundError: No module named 'matplotlib'

##### Agora vamos ver o faker

In [6]:
fake = Faker(locale='pt-BR')
data_atual = date.today()
porcentagem_comissao = 0.015

nota_fiscal = []
nome = []
cpf = []
cidade = []
profissao = []
endereco = []
email = []
data_nascimento = []
idade = []
data_emissao = []
valor_nota = []
comissao = []
id_vendedor = []
vendedores = []
vendedores_id = []
vendedores_cpf = []
vendedores_endereco = []
comissao_float = []

for _ in range(10):
    vendedores_id.append(fake.random_number(digits=4))
    vendedores.append(fake.name())
    vendedores_cpf.append(fake.cpf())
    vendedores_endereco.append(fake.address())

for i in range(2500):
    nota_fiscal.append(fake.random_number(digits=6, fix_len=True))
    nome.append(fake.name())
    cpf.append(fake.cpf())
    cidade.append(fake.city())
    profissao.append(fake.job())
    endereco.append(fake.street_address())
    email.append(fake.email())

    data = fake.date_of_birth(minimum_age=18)
    data_nascimento.append(data.strftime('%d/%m/%Y'))
    
    idade.append(data_atual.year - data.year)
    data = data_atual - timedelta(days=fake.random_int(min=0, max=365))
    data_emissao.append(data.strftime('%d/%m/%Y'))

    vlr = fake.pyfloat(left_digits=3, right_digits=2, positive=True)
    valor_nota.append("R$" + str(vlr).replace(".", ','))
    comissao.append("R$"+ str(round(vlr * porcentagem_comissao, 2)).replace(".", ","))

    comissao_float.append(round(vlr * porcentagem_comissao, 2))

    id_vendedor.append(fake.random_choices(vendedores_id, 1)[0])

# Geração do DataFrame

df_vendas_bruto = pd.DataFrame(
    {
        'NOTA_FISCAL': nota_fiscal,
        'NOME': nome,
        'CPF': cpf,
        'CIDADE': cidade,
        'CARGO': profissao,
        'ENDERECO': endereco,
        'EMAIL': email,
        'DATA_NASCIMENTO': data_nascimento,
        'IDADE': idade,
        'DATA_EMISSAO': data_emissao,
        'VALOR_NOTA': valor_nota,
        'COMISSAO': comissao,
        'COMISSAO_FLOAT': comissao_float,
        'VENDEDOR': id_vendedor
    }
)

df_vendedores = pd.DataFrame(
    {
        'ID_VENDEDOR': vendedores_id,
        'NOME_VENDEDOR': vendedores,
        'CPF_VENDEDOR': vendedores_cpf,
        'ENDERECO_VENDEDOR': vendedores_endereco
    }
)

os.makedirs('.\\output', exist_ok=True)
df_vendas_bruto.to_csv(r'.\output\Vendas (Bruto)_falsa.csv', index=False)
df_vendedores.to_csv(r'.\output\Vendedores_falsos.csv', index=False)

print('Arquivo gerado com sucesso...')

Arquivo gerado com sucesso...


#### Agora vamos ver o num2words

In [8]:

df_comissao_por_vendedor = df_vendas_bruto.groupby('VENDEDOR').agg({'COMISSAO_FLOAT': 'sum'}).reset_index()
df_comissao_por_vendedor = df_comissao_por_vendedor.merge(df_vendedores, left_on='VENDEDOR', right_on='ID_VENDEDOR', how='left')
df_comissao_por_vendedor = df_comissao_por_vendedor[['VENDEDOR', 'NOME_VENDEDOR', 'COMISSAO_FLOAT']]

for index, row in df_comissao_por_vendedor.iterrows():
    comissao_str = row['COMISSAO_FLOAT']
    comissao_float = round(float(comissao_str), 2)
    df_comissao_por_vendedor.at[index, 'COMISSAO_EXTENSO'] = nw(comissao_float, lang='pt_BR', to='currency')
    
df_comissao_por_vendedor.to_csv(r'.\output\Comissao_por_vendedor.csv', index=False)

print('Arquivo de comissão por vendedor gerado com sucesso...')

print(nw(number=123456.78, lang='pt_BR', to='currency'))  # Exemplo de uso do num2words

Arquivo de comissão por vendedor gerado com sucesso...
cento e vinte e três mil, quatrocentos e cinquenta e seis reais e setenta e oito centavos


##### Agora vamos ver o Numpy

In [9]:
np.random.seed(40)
vendas_mensal = np.random.randint(2000, 4000, size=12)
aumento_percentual = np.diff(vendas_mensal) / vendas_mensal[-1] * 100
mes = np.array(['Janeiro', 'Fevereiro', 'Março', 'Abril', 'Maio', 'Junho', 'Julho', 'Agosto', 'Setembro', 'Outubro', 'Novembro', 'Dezembro'])

for i, crescimento in enumerate(aumento_percentual):
    print(f'do mes de {mes[i]} para o mes de {mes[i+1]} teve um crescimento {crescimento:.2f}%')

print(f'a média das vendas do ano foi {np.mean(vendas_mensal):.2f}')
print(f'A maior venda foi {vendas_mensal.max()}')
print(f'venda em ultima hora, o ultimo teve mais {nw(6000, lang='pt_BR', to='currency')} de venda')
vendas_mensal[-1] = vendas_mensal[-1] + 6000

desvio_vendas = np.std(vendas_mensal)
print(f'o desvio padrão das vendas é {desvio_vendas:.2f} R$')

do mes de Janeiro para o mes de Fevereiro teve um crescimento -31.81%
do mes de Fevereiro para o mes de Março teve um crescimento -5.96%
do mes de Março para o mes de Abril teve um crescimento 4.44%
do mes de Abril para o mes de Maio teve um crescimento 23.94%
do mes de Maio para o mes de Junho teve um crescimento 12.60%
do mes de Junho para o mes de Julho teve um crescimento -3.77%
do mes de Julho para o mes de Agosto teve um crescimento 11.22%
do mes de Agosto para o mes de Setembro teve um crescimento -12.29%
do mes de Setembro para o mes de Outubro teve um crescimento 16.06%
do mes de Outubro para o mes de Novembro teve um crescimento -34.80%
do mes de Novembro para o mes de Dezembro teve um crescimento 26.13%
a média das vendas do ano foi 3051.33
A maior venda foi 3863
venda em ultima hora, o ultimo teve mais seis mil reais de venda
o desvio padrão das vendas é 1905.30 R$


In [16]:
nomes = np.array([fake.name(), fake.name(), fake.name(), fake.name(), fake.name(), fake.name(), fake.name(), fake.name(), fake.name(), fake.name()])
vlr = np.array([50, 100, 150, 100, 5000, 140,120, 322, 5125, 102])

media_vlr = np.mean(vlr)
desvio_vlr = np.std(vlr)
fora_padrao = np.abs(vlr - media_vlr) > 2 * desvio_vlr

for nome, vlr in zip(nomes[fora_padrao], vlr[fora_padrao]):
    print(f'Está fora do padrao: {nome} - {vlr}\n')

lista = np.array([np.linspace(10, 50, 5), np.linspace(15, 30, 5), np.linspace(8, 60, 5), np.linspace(10, 50, 5)])
print(lista)
print(f'Média por coluna: {lista.mean(axis=0)}, Média por linha: {lista.mean(axis=1)}, {type(lista.mean(axis=0))}')
print(f'Número de dimensões: {lista.ndim}')
print(f'Tamanho total: {lista.size}')
print(f'Tipo de dado: {lista.dtype}')
print(f'Elementos menores que 30: {lista[lista < 30]}')
print(f'Transposta: \n{lista.T}')
print(f'Produto escalar: \n{np.dot(lista, lista.T)}')
print(f'Reshape: \n{lista.reshape(4, 5)}')
print(type(nomes))

Está fora do padrao: Olivia Vasconcelos - 5125

[[10.   20.   30.   40.   50.  ]
 [15.   18.75 22.5  26.25 30.  ]
 [ 8.   21.   34.   47.   60.  ]
 [10.   20.   30.   40.   50.  ]]
Média por coluna: [10.75   19.9375 29.125  38.3125 47.5   ], Média por linha: [30.  22.5 34.  30. ], <class 'numpy.ndarray'>
Número de dimensões: 2
Tamanho total: 20
Tipo de dado: float64
Elementos menores que 30: [10.   20.   15.   18.75 22.5  26.25  8.   21.   10.   20.  ]
Transposta: 
[[10.   15.    8.   10.  ]
 [20.   18.75 21.   20.  ]
 [30.   22.5  34.   30.  ]
 [40.   26.25 47.   40.  ]
 [50.   30.   60.   50.  ]]
Produto escalar: 
[[5500.    3750.    6400.    5500.   ]
 [3750.    2671.875 4312.5   3750.   ]
 [6400.    4312.5   7470.    6400.   ]
 [5500.    3750.    6400.    5500.   ]]
Reshape: 
[[10.   20.   30.   40.   50.  ]
 [15.   18.75 22.5  26.25 30.  ]
 [ 8.   21.   34.   47.   60.  ]
 [10.   20.   30.   40.   50.  ]]
<class 'numpy.ndarray'>


In [82]:
vlrs = np.array([[1, 2, 3, 4, 5], np.zeros(5), np.ones(5)])
arange_result = np.arange(start=0, stop=10, step=2)
lins = np.linspace(start=0, stop=10, num=5)
print(vlrs)
print(type(np.mean(vlrs)))
print(type(np.abs(vlrs)))
print(np.abs(vlrs))
print(type(np.size(vlrs)))
print(np.mean(vlrs, axis=0))

[[1. 2. 3. 4. 5.]
 [0. 0. 0. 0. 0.]
 [1. 1. 1. 1. 1.]]
<class 'numpy.float64'>
<class 'numpy.ndarray'>
[[1. 2. 3. 4. 5.]
 [0. 0. 0. 0. 0.]
 [1. 1. 1. 1. 1.]]
<class 'int'>
[0.66666667 1.         1.33333333 1.66666667 2.        ]
